# 03 — Coorte, painel, atributos e divisão temporal (seções 6–11 do v5)

**Lê:** `weekly_observed`, `weeks_info`, `corte.json` (01); `exogenas_semanais`, `exogenas_listas.json` (02).  
**Grava:** `coorte`, `panel_base`, `painel` (com a coluna `conjunto` = treino/teste/fora), `painel_spec.json`, `divisao.json`.

Este notebook **constrói** o teste mas não avalia nada nele. Daqui em diante, as linhas de teste só saem por `A.carregar_teste()`, que exige a seleção congelada.

In [4]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "03_painel"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
AVISO de reprodutibilidade: pandas=3.1.1; referência recomendada=2.2.3
AVISO de reprodutibilidade: numpy=2.5.4; referência recomendada=2.1.3
AVISO de reprodutibilidade: scikit-learn=1.9.1; referência recomendada=1.6.1
Notebook: 03_painel | execução: v6_modular | raiz: C:\Users\andre\OneDrive\Documentos\Projetos\Tccv2\tcc_fluxo_modular_v6


In [ ]:
from tcc_fluxo import painel as P

corte = A.carregar_json("corte")
TEST_END = pd.Timestamp(corte["TEST_END"])
weekly_observed = A.carregar_df("weekly_observed")
wi = A.carregar_df("weeks_info").set_index("week_start")
exog = A.carregar_df("exogenas_semanais")
listas = A.carregar_json("exogenas_listas")

print("Semanas com aprovações antes de/até o início do painel:")
display(wi.loc[:C.PANEL_START + pd.Timedelta(weeks=3), ["pedidos", "valor"]])
pre = weekly_observed["week_start"] < C.PANEL_START
print(f"Registros vendedor-semana anteriores a {C.PANEL_START.date()} descartados: {pre.sum():,}")

## 6. Coorte (somente informação até `TRAIN_END`)

In [ ]:
elig, cohort, COHORT_LABEL, observed_train = P.coorte(
    weekly_observed, C.PANEL_START, C.TRAIN_END, C.MIN_CALENDAR_WEEKS, C.MIN_SALES_WEEKS,
    C.USE_MEI_PROXY, C.MEI_MONTHLY_PROXY_LIMIT)
print("Critério:", COHORT_LABEL)
print(f"Vendedores com venda no treino: {len(elig):,} | elegíveis: {len(cohort):,} | excluídos: {len(elig) - len(cohort):,}")
fora = set(weekly_observed.loc[weekly_observed["week_start"] > C.TRAIN_END, "seller_id"]) - set(elig.index)
print(f"Vendedores com venda no teste e sem venda no treino (fora da coorte por construção): {len(fora):,}")
display(cohort[["avg_weekly_revenue_train", "avg_monthly_revenue_train", "revenue_train", "sales_weeks",
                "calendar_weeks"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T)

## 7. Painel vendedor-semana (zeros preenchidos, sem interpolação)

In [ ]:
panel_base = P.montar_panel_base(cohort, weekly_observed, TEST_END, C.PANEL_START)
print(f"Linhas: {len(panel_base):,} | vendedores: {panel_base['seller_id'].nunique():,} | "
      f"semanas: {panel_base['week_start'].nunique()}")
print(f"Semanas com receita zero: {panel_base['revenue'].eq(0).mean():.1%}")

## 8–10. Atributos sem vazamento, calendário da semana-alvo e alvo (h = 1 semana)

In [ ]:
panel = P.attach_exogenous_features(panel_base, exog)
EXOG = listas["EXOGENOUS_FEATURES"]
ALL_NUMERIC = P.BASE_NUMERIC_FEATURES + EXOG
assert ((panel["target_week"] - panel["week_start"]).dropna() == pd.Timedelta(days=7)).all()
assert panel.loc[panel["target_week"].notna(), EXOG].notna().all().all()

feature_cols = ALL_NUMERIC + ["target_week", "target_week_of_year"]
_, changed = P.teste_vazamento(panel_base, panel, exog, feature_cols, C.LEAK_TEST_CUT, C.RANDOM_SEED)
print(f"Atributos até {C.LEAK_TEST_CUT.date()} idênticos após perturbar o futuro: OK "
      f"(controle: {changed:.0%} dos alvos após o corte mudaram)")

## 11. Divisão temporal pela data do alvo

In [ ]:
painel, resumo = P.dividir(panel, ALL_NUMERIC, C.TRAIN_TARGET_END, TEST_END)
print(pd.Series(resumo))
profile = P.perfis_treino(cohort)
print(profile.groupby("perfil", observed=True)["freq_venda_treino"].agg(["min", "max", "size"]))

In [ ]:
A.salvar_df(cohort.reset_index(), "coorte")
A.salvar_df(profile.reset_index(), "perfis_treino")
A.salvar_df(panel_base, "panel_base")
A.salvar_df(painel, "painel")
A.salvar_json({"BASE_NUMERIC_FEATURES": P.BASE_NUMERIC_FEATURES, "EXOGENOUS_FEATURES": EXOG,
               "CALENDAR_FEATURES": listas["CALENDAR_FEATURES"], "HOLIDAY_FEATURES": listas["HOLIDAY_FEATURES"],
               "COMMERCIAL_FEATURES": listas["COMMERCIAL_FEATURES"], "MACRO_FEATURES": listas["MACRO_FEATURES"],
               "CAT_FEATURES": P.CAT_FEATURES, "TARGET": P.TARGET, "COHORT_LABEL": COHORT_LABEL}, "painel_spec")
A.salvar_json({**resumo, "TEST_END": TEST_END, "teste_vazamento_corte": C.LEAK_TEST_CUT,
               "teste_vazamento_controle": changed}, "divisao")

### 6.1 Porte da coorte (descritivo — não exclui ninguém)
Quantos vendedores passam do limite de faturamento do MEI, usando a receita média mensal do treino
(proporcional aos meses de atividade em 2017). A receita é a proxy do alvo, não o faturamento fiscal.
Esta célula só lê o artefato `coorte`: pode ser rodada sozinha (depois da célula de setup) sem refazer o painel.

In [5]:
from tcc_fluxo import painel as P
coorte_art = A.carregar_df("coorte")
LIMITES_MEI = {"MEI a partir de 2018 (R$ 81 mil/ano)": C.MEI_MONTHLY_PROXY_LIMIT,   # LC 155/2016
               "MEI em 2017 (R$ 60 mil/ano)": 5000.0}                              # limite vigente nos dados de treino
porte = P.resumo_porte(coorte_art, LIMITES_MEI)
A.tabela(porte, "porte_coorte_mei.csv"); display(porte)
A.salvar_json(porte.to_dict("records"), "porte_coorte")

,limite,limite_mensal_R$,vendedores_acima,pct_coorte,n_coorte
0,MEI a partir de 2018 (R$ 81 mil/ano),"6,750.0000",16,0.0184,870
1,MEI em 2017 (R$ 60 mil/ano),"5,000.0000",24,0.0276,870


{'arquivo': 'porte_coorte.json',
 'sha256': '415706deffe9178b57f707acc3d17d47f0f53a88a1b238c8df0de90377719c59',
 'notebook': '03_painel',
 'criado_em': '2026-11-15T13:12:44',
 'entradas': {'coorte': 'a87c7af0c87a80f3c9872e4f6a2deab4ffcf633580b85fd136de67e8b04f9a48'}}

In [3]:
A.finalizar()

03_painel: leu 1 artefato(s), gravou 0.
